In [2]:
import pandas as pd
from fbprophet import Prophet
import datetime as dt
import numpy as np
import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings(action='once')
import logging, sys
logging.disable(sys.maxsize)


In [3]:
df = pd.read_csv('C:\\Users\\USER\\Desktop\\BABE\\DATA\\Lakshmi2019New.csv', encoding='iso-8859-1')
df.columns.values
unicity = df.CITY.unique()
unicity

In [8]:
#prediction on 2018 & 2019
df['ds'] = pd.to_datetime(df['ds'])
maininputall = df
#exclude2018 = df[df['ds'].dt.year != 2018]
include2018 = df[df['ds'].dt.year == 2018]
include2019 = df[df['ds'].dt.year == 2019]
#exclude2018 = exclude2018[exclude2018['ds'].dt.year != 2016]
#exclude2018 = exclude2018[exclude2018['ds'].dt.year != 2018]
exclude2017 = df[df['ds'].dt.year != 2017]
exclude2017


In [5]:
def drange(start, stop, step):
    while start < stop:
            yield start
            start += step
            


In [12]:
##### prediction on 2019 & 2020 MPD
columns=['CITY','PREDICTED2020','PREDICTED2019' ,'ACTUAL2018','ACTUAL2019','MPD2019', 'MPD2020','DEV' ]
columnsnew=['DEV','pred2019', 'pred2020']
outputfinal = pd.DataFrame( columns=columns)
outputnew = pd.DataFrame( columns=columns)
output = pd.DataFrame( columns=columns)
devout = pd.DataFrame( columns=columnsnew)
#if(i=="?eskÃ½ Krumlov"):
i='Berlin'
try:

            predexcl2019 =""
            predincl2019=""
            pred=""
            MPDincl2017=""
            MPDExcl2017=""
            act2018=""
            pred2019=""
            actincl2018=""
            MPDincl2017FLAG = "Y"
            DEV = ""
            maininputall = maininputall[maininputall['CITY'] ==  i ]

            include2018City = include2018[include2018['CITY'] ==  i ]
            include2019City = include2019[include2019['CITY'] ==  i ]
#             exclude2017 = df[df['ds'].dt.year != 2017]
#             exclude2017 = exclude2017[exclude2017['CITY'] ==  i ]
            countexck = maininputall['ds'].count()
            #print(countexck, len(include2018City))
            if(countexck> 10 and len(include2018City) >0 ):
                for chngpriscale in drange(0.01,0.10,0.02):

                    m = Prophet(changepoint_prior_scale=chngpriscale)
                    #maininputall['cap'] = 8.5
                    m.fit(maininputall)
                    future = m.make_future_dataframe(periods = 24, freq = 'M')
                    future.tail()
                    forecast = m.predict(future)
                    per = forecast.ds.dt.to_period("Y")
                    g = forecast.groupby(per)
                    pred = g.mean()
                    print(pred)
                    #print(pred)
                    #print(pred)
                   # m.plot(forecast)
                    if(len(include2019City) ) >0:
#                                 predincl2019 = ( (pred['yhat_lower'][pred['yhat'].count()-1] ) + pred['yhat'][pred['yhat'].count()-1])/2
#                                 predincl2020 = ( (pred['yhat_lower'][pred['yhat'].count()-1] ) + pred['yhat'][pred['yhat'].count()-1])/2
#                                 #print(predincl2019,predincl2020)
                        predincl2019 =  (pred['yhat_lower'][pred['yhat'].count()-1] ) 
                        predincl2020 =  (pred['yhat'][pred['yhat'].count()-2] ) 


                    else:
                        predincl2019 = pred['yhat_lower'][pred['yhat'].count()-2] 
                        predincl2020 = pred['yhat'][pred['yhat'].count()-3] 

                    #print(i)  
    #                 predincl2020 = pred['yhat'][4]
    #                 predincl2019 = pred['yhat'][3]
                    actincl2018 = include2018City['y'].mean()
                    if(len(include2019City) >0):
                        actincl2019 = include2019City['y'].mean()
                    else:
                        actincl2019 = 0

                    DEV = abs(predincl2019 - actincl2019)
                    print(DEV)
                    #predincl2019 = ((predincl2019+actincl2019)/2)
                    MPDincl2019 = (predincl2019 - actincl2018)*100/actincl2018
                    MPDincl2020 = (predincl2020 - actincl2019)*100/actincl2019
                    #predincl2020 = actincl2018 * (MPDincl2020/100) + actincl2018

#                                 predincl2019 = 100 + (actincl2019*MPDincl2020)
#                                 DEV = abs(predincl2019 - actincl2019)
#                                 MPDincl2020 = (predincl2020 - actincl2019)*100/actincl2019
                    #calculating the MPD for the year 2018

    #                 outoutlist = [i, predincl2020 ,predincl2019, MPDincl2020, MPDincl2019]

                    outoutlist = [i, predincl2020, predincl2019, actincl2018,actincl2019, MPDincl2019, MPDincl2020, float(DEV)]
                    #print(outoutlist)
                    output = pd.DataFrame(np.array([outoutlist]), columns=['CITY','PREDICTED2020','PREDICTED2019' ,'ACTUAL2018','ACTUAL2019','MPD2019', 'MPD2020' ,'DEV' ]).append(output, ignore_index=True)
                    devout = [float(DEV),float(predincl2019),float(predincl2020)]
                    #print(outoutlist)


                #else:
                    #print(len(include2018City))
                    #print(len(include2019City))
            output.DEV = output.DEV.astype(float)


            output = output[output.DEV == output['DEV'].min()]

            outputdf = pd.DataFrame(output)
            if(outputdf.empty):
                print("EMPTY DF for ", i)
            else:
                if( float(outputdf['ACTUAL2019']) == 0):
                    output['MPD2020'] = (float(output['PREDICTED2020']) - float(output['PREDICTED2019']))*100/float(output['PREDICTED2019'])

                elif ( float(output['ACTUAL2019']) > 0 and
                      ( float(output['DEV']) >= 10 or  float(output['DEV']) <= -10) or MPDincl2019 >=7 or MPDincl2019 <=-7
                       ) :
                    #print(predincl2019)

#                         predincl2019 = (pred['yhat_lower'][pred['yhat_lower'].count()-1] +pred['yhat'][pred['yhat'].count()-1] )/2
#                         predincl2020 = (pred['yhat_lower'][pred['yhat_lower'].count()-2] +pred['yhat'][pred['yhat'].count()-1])/2
#                         MPDincl2019 = (predincl2019 - actincl2018)*100/actincl2018
#                         MPDincl2020 = (predincl2020 - predincl2019)*100/predincl2019
#                         output['MPD2020'] = MPDincl2020
#                         output['MPD2019'] = MPDincl2019
#                         if(MPDincl2019 >=5 or  MPDincl2019 <=-5) :
                    #print(devout)
                    output['PREDICTED2019'] = (float(output['ACTUAL2019']) + float(output['PREDICTED2019']) )/2
                    output['MPD2020'] = (float(output['PREDICTED2020'] )- float(output['ACTUAL2019']) )*100/float(output['ACTUAL2019'])
                    output['MPD2019'] = (float(output['PREDICTED2019']) -float(output['ACTUAL2018']) )*100/float(output['ACTUAL2018'])
                    output['DEV'] = abs(float(output['PREDICTED2019']) - float(output['ACTUAL2019']))
                    output['MPD2020'] = (float(output['PREDICTED2020'] )- float(output['ACTUAL2019']) )*100/float(output['ACTUAL2019'])


                outputnew= output.values.tolist()
                print(outputnew)
                #print(output)
                #outputfinal.append(output)
                outputfinal = pd.DataFrame(output, columns=['CITY','PREDICTED2020','PREDICTED2019' ,'ACTUAL2018','ACTUAL2019','MPD2019', 'MPD2020' ,'DEV' ]).append(outputfinal, ignore_index=True)
                #print(outputfinal)
                output = ""



except AssertionError as error:
    print("Exception at " + i + " error " + error)
print("END")






In [ ]:
##### prediction on 2019 & 2020 MPD
columns=['CITY','PREDICTED2020','PREDICTED2019' ,'ACTUAL2018','ACTUAL2019','MPD2019', 'MPD2020','DEV' ]
columnsnew=['DEV','pred2019', 'pred2020']
outputfinal = pd.DataFrame( columns=columns)
chngpriscale = 0.02
for i in unicity:
    outputnew = pd.DataFrame( columns=columns)
    output = pd.DataFrame( columns=columns)
    devout = pd.DataFrame( columns=columnsnew)
    #if(i=="?eskÃ½ Krumlov"):
    try:

                predexcl2019 =""
                predincl2019=""
                pred=""
                MPDincl2017=""
                MPDExcl2017=""
                act2018=""
                pred2019=""
                actincl2018=""
                MPDincl2017FLAG = "Y"
                DEV = ""
                maininputall = exclude2017[exclude2017['CITY'] ==  i ]

                include2018City = include2018[include2018['CITY'] ==  i ]
                include2019City = include2019[include2019['CITY'] ==  i ]
    #             exclude2017 = df[df['ds'].dt.year != 2017]
    #             exclude2017 = exclude2017[exclude2017['CITY'] ==  i ]
                countexck = maininputall['ds'].count()
                #print(countexck, len(include2018City))
                if(countexck> 10 and len(include2018City) >0 ):
                    for chngpriscale in drange(0.01,0.10,0.02):

                        m = Prophet(changepoint_prior_scale=chngpriscale)
                        #maininputall['cap'] = 8.5
                        m.fit(maininputall)
                        future = m.make_future_dataframe(periods = 24, freq = 'M')
                        future.tail()
                        forecast = m.predict(future)
                        per = forecast.ds.dt.to_period("Y")
                        g = forecast.groupby(per)
                        pred = g.mean()
                        #print(pred)
                        #print(pred)
                       # m.plot(forecast)
                        if(len(include2019City) ) >0:
#                                 predincl2019 = ( (pred['yhat_lower'][pred['yhat'].count()-1] ) + pred['yhat'][pred['yhat'].count()-1])/2
#                                 predincl2020 = ( (pred['yhat_lower'][pred['yhat'].count()-1] ) + pred['yhat'][pred['yhat'].count()-1])/2
#                                 #print(predincl2019,predincl2020)
                            predincl2019 =  (pred['yhat_lower'][pred['yhat'].count()-1] ) 
                            predincl2020 =  (pred['yhat'][pred['yhat'].count()-2] ) 


                        else:
                            predincl2019 = pred['yhat_lower'][pred['yhat'].count()-2] 
                            predincl2020 = pred['yhat'][pred['yhat'].count()-3] 

                        #print(i)  
        #                 predincl2020 = pred['yhat'][4]
        #                 predincl2019 = pred['yhat'][3]
                        actincl2018 = include2018City['y'].mean()
                        if(len(include2019City) >0):
                            actincl2019 = include2019City['y'].mean()
                        else:
                            actincl2019 = 0

                        DEV = abs(predincl2019 - actincl2019)
                        #predincl2019 = ((predincl2019+actincl2019)/2)
                        MPDincl2019 = (predincl2019 - actincl2018)*100/actincl2018
                        MPDincl2020 = (predincl2020 - actincl2018)*100/actincl2018
                        #predincl2020 = actincl2018 * (MPDincl2020/100) + actincl2018

#                                 predincl2019 = 100 + (actincl2019*MPDincl2020)
#                                 DEV = abs(predincl2019 - actincl2019)
#                                 MPDincl2020 = (predincl2020 - actincl2019)*100/actincl2019
                        #calculating the MPD for the year 2018

        #                 outoutlist = [i, predincl2020 ,predincl2019, MPDincl2020, MPDincl2019]

                        outoutlist = [i, predincl2020, predincl2019, actincl2018,actincl2019, MPDincl2019, MPDincl2020, float(DEV)]
                        #print(outoutlist)
                        output = pd.DataFrame(np.array([outoutlist]), columns=['CITY','PREDICTED2020','PREDICTED2019' ,'ACTUAL2018','ACTUAL2019','MPD2019', 'MPD2020' ,'DEV' ]).append(output, ignore_index=True)
                        devout = [float(DEV),float(predincl2019),float(predincl2020)]
                        #print(outoutlist)


                    #else:
                        #print(len(include2018City))
                        #print(len(include2019City))
                output.DEV = output.DEV.astype(float)


                output = output[output.DEV == output['DEV'].min()]

                outputdf = pd.DataFrame(output)
                if(outputdf.empty):
                    print("EMPTY DF for ", i)
                else:
                    if( float(outputdf['ACTUAL2019']) == 0):
                        output['MPD2020'] = (float(output['PREDICTED2020']) - float(output['PREDICTED2019']))*100/float(output['PREDICTED2019'])

                    elif ( float(output['ACTUAL2019']) > 0 and
                          ( float(output['DEV']) >= 10 or  float(output['DEV']) <= -10) or MPDincl2019 >=7 or MPDincl2019 <=-7
                           ) :
                        #print(predincl2019)

#                         predincl2019 = (pred['yhat_lower'][pred['yhat_lower'].count()-1] +pred['yhat'][pred['yhat'].count()-1] )/2
#                         predincl2020 = (pred['yhat_lower'][pred['yhat_lower'].count()-2] +pred['yhat'][pred['yhat'].count()-1])/2
#                         MPDincl2019 = (predincl2019 - actincl2018)*100/actincl2018
#                         MPDincl2020 = (predincl2020 - predincl2019)*100/predincl2019
#                         output['MPD2020'] = MPDincl2020
#                         output['MPD2019'] = MPDincl2019
#                         if(MPDincl2019 >=5 or  MPDincl2019 <=-5) :
                        #print(devout)
                        output['PREDICTED2019'] = (float(output['ACTUAL2019']) + float(output['PREDICTED2019']) )/2
                        output['MPD2020'] = (float(output['PREDICTED2020'] )- float(output['ACTUAL2018']) )*100/float(output['ACTUAL2018'])
                        output['MPD2019'] = (float(output['PREDICTED2019']) -float(output['ACTUAL2018']) )*100/float(output['ACTUAL2018'])
                        output['DEV'] = abs(float(output['PREDICTED2019']) - float(output['ACTUAL2019']))
                        output['MPD2020'] = (float(output['PREDICTED2020'] )- float(output['ACTUAL2018']) )*100/float(output['ACTUAL2018'])


                    outputnew= output.values.tolist()
                    print(outputnew)
                    #print(output)
                    #outputfinal.append(output)
                    outputfinal = pd.DataFrame(output, columns=['CITY','PREDICTED2020','PREDICTED2019' ,'ACTUAL2018','ACTUAL2019','MPD2019', 'MPD2020' ,'DEV' ]).append(outputfinal, ignore_index=True)
                    #print(outputfinal)
                    output = ""



    except AssertionError as error:
        print("Exception at " + i + " error " + error)
        continue
print("END")




